In [6]:
# Learning how the BPE algorithm works in practice
# Author: Devansh Bisla
# Date: 09/30/2023

# Tokenization algorithm involves three steps:
# 1. Normalization: removal of any accents or unneccessary white spaces etc
# 2. Pre-Tokenization: Splitting the text into words, sometimes we add another token in-place of white space etc.
# 3. Tokenization process: BPE

### Why BPE, why not characters? 
You can use characters, and many models do (or mix char + other signals). People add BPE (or similar) because it trades off a few things:

### Why pure characters are attractive

Small, fixed vocabulary — one per Unicode code point (or byte for byte-level BPE), so almost no unknown tokens.
No segmentation surprises — you never “split” a rare word wrong; you spell it out.
Good for morphology — prefixes/suffixes are explicit.
Why people often prefer subwords (BPE, SentencePiece, etc.)

Sequence length — A word is 1 subword token vs many character tokens. Longer sequences mean more compute and memory for transformers (attention is roughly quadratic in length).
Meaningful chunks — Subwords often align with morphemes or frequent fragments (ing, tion), so the model can reuse patterns without spelling every letter every time.
Data efficiency — With limited data, learning from character streams alone can be slower; frequent n-grams give reusable “words” without needing a huge word-level vocab.
In short: Characters minimize vocabulary and OOV issues but inflate sequence length and can make learning slower. BPE keeps a bounded vocab and shorter sequences than pure characters, while still handling rare/unknown words by composing pieces instead of memorizing whole words.

So it’s not “wrong” to use characters—it’s a design choice between simplicity and sequence length vs efficiency and statistical strength for your model and dataset.

In [8]:
from collections import defaultdict, Counter

corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]


# Normalization
# Pre-Tokenization
pre_tokenized_corpus = []
for sentence in corpus:
    sentence = sentence.replace(' ', 'Ġ')
    sentence = ['Ġ' + s for s in sentence.split('Ġ')]
    sentence[0] = sentence[0].lstrip('Ġ')
    pre_tokenized_corpus += sentence

word_freqs = Counter(pre_tokenized_corpus)

print(pre_tokenized_corpus)
print(word_freqs)


['This', 'Ġis', 'Ġthe', 'ĠHugging', 'ĠFace', 'ĠCourse.', 'This', 'Ġchapter', 'Ġis', 'Ġabout', 'Ġtokenization.', 'This', 'Ġsection', 'Ġshows', 'Ġseveral', 'Ġtokenizer', 'Ġalgorithms.', 'Hopefully,', 'Ġyou', 'Ġwill', 'Ġbe', 'Ġable', 'Ġto', 'Ġunderstand', 'Ġhow', 'Ġthey', 'Ġare', 'Ġtrained', 'Ġand', 'Ġgenerate', 'Ġtokens.']
Counter({'This': 3, 'Ġis': 2, 'Ġthe': 1, 'ĠHugging': 1, 'ĠFace': 1, 'ĠCourse.': 1, 'Ġchapter': 1, 'Ġabout': 1, 'Ġtokenization.': 1, 'Ġsection': 1, 'Ġshows': 1, 'Ġseveral': 1, 'Ġtokenizer': 1, 'Ġalgorithms.': 1, 'Hopefully,': 1, 'Ġyou': 1, 'Ġwill': 1, 'Ġbe': 1, 'Ġable': 1, 'Ġto': 1, 'Ġunderstand': 1, 'Ġhow': 1, 'Ġthey': 1, 'Ġare': 1, 'Ġtrained': 1, 'Ġand': 1, 'Ġgenerate': 1, 'Ġtokens.': 1})


In [9]:
# Tokenizatiion Process: BPE
# 1. Create a vocab i.e set of all the words 
vocab = set()
for words in pre_tokenized_corpus:
    vocab = vocab | set(words)
vocab = list(vocab)
vocab += ["<eos>"]
print(f"Length of vocab: {len(vocab)}")

# 2. Start training BPE - model
# 2a. Find most common pair of characters in the corpus
# 2b. Merge the common pairs 
# 2c Repeat untill you get vocab size

def compute_pair_freqs(splits):
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1:
            continue
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            pair_freqs[pair] += freq
    return pair_freqs

def merge_pairs(a, b, splits):
    for word, split in splits.items():
        if len(split) == 1:
            continue

        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                split = split[:i] + [a + b] + split[i+2:]
            else:
                i+=1
        splits[word] = split
    return splits

vocab_size = 50
# spits is a dict where each key is a word and the value is a list of characters
# e.g {"this": ["t", "h", "i", "s"], "is": ["i", "s"], "the": ["t", "h", "e"], "and": ["a", "n", "d"]}
splits = {word: [c for c in word] for word in pre_tokenized_corpus} 
# merges is a dict where each key is a pair of characters and the value is the merged character
merges = {}

while len(vocab) < vocab_size:
    # compute the frequency of each pair of characters in the corpus e.g {"th": 1 , "hi": 1, "is": 2, "s ": 1}
    pair_freqs = compute_pair_freqs(splits)
    
    best_pair = max(pair_freqs, key=pair_freqs.get)
    merges[best_pair[0], best_pair[1]] = ''.join(best_pair)
    
    vocab.append(''.join(best_pair))

    # merge the best pair in the splits
    # this function takes in a pair of characters and the splits and returns a new splits
    splits = merge_pairs(*best_pair, splits)
 
print(merges)
print(f"Length of vocab (post training BPE): {len(vocab)}")

Length of vocab: 31
{('Ġ', 't'): 'Ġt', ('i', 's'): 'is', ('e', 'r'): 'er', ('Ġ', 'a'): 'Ġa', ('Ġt', 'o'): 'Ġto', ('e', 'n'): 'en', ('T', 'h'): 'Th', ('Th', 'is'): 'This', ('o', 'u'): 'ou', ('s', 'e'): 'se', ('Ġto', 'k'): 'Ġtok', ('Ġtok', 'en'): 'Ġtoken', ('n', 'd'): 'nd', ('Ġ', 'is'): 'Ġis', ('Ġt', 'h'): 'Ġth', ('Ġth', 'e'): 'Ġthe', ('i', 'n'): 'in', ('Ġa', 'b'): 'Ġab', ('Ġtoken', 'i'): 'Ġtokeni'}
Length of vocab (post training BPE): 50


In [10]:
def tokenize(text):
    text = text.replace(' ', 'Ġ')
    text = ['Ġ' + s for s in text.split('Ġ')]
    text[0] = text[0].lstrip("Ġ")
    print(text)

    splits = [[l for l in word] for word in text]
    for pair, merge in merges.items():
        for idx, split in enumerate(splits):
            i = 0
            while i < len(split) - 1:
                if split[i] == pair[0] and split[i + 1] == pair[1]:
                    split = split[:i] + [merge] + split[i + 2 :]
                else:
                    i += 1
            splits[idx] = split

    return sum(splits, [])

In [11]:
tokenize("This is not a token.")

['This', 'Ġis', 'Ġnot', 'Ġa', 'Ġtoken.']


['This', 'Ġis', 'Ġ', 'n', 'o', 't', 'Ġa', 'Ġtoken', '.']

In [12]:
merges

{('Ġ', 't'): 'Ġt',
 ('i', 's'): 'is',
 ('e', 'r'): 'er',
 ('Ġ', 'a'): 'Ġa',
 ('Ġt', 'o'): 'Ġto',
 ('e', 'n'): 'en',
 ('T', 'h'): 'Th',
 ('Th', 'is'): 'This',
 ('o', 'u'): 'ou',
 ('s', 'e'): 'se',
 ('Ġto', 'k'): 'Ġtok',
 ('Ġtok', 'en'): 'Ġtoken',
 ('n', 'd'): 'nd',
 ('Ġ', 'is'): 'Ġis',
 ('Ġt', 'h'): 'Ġth',
 ('Ġth', 'e'): 'Ġthe',
 ('i', 'n'): 'in',
 ('Ġa', 'b'): 'Ġab',
 ('Ġtoken', 'i'): 'Ġtokeni'}